# Step 7b — Multi-corpus scaler + label encoding



In [ ]:
#!pip install -q scikit-learn==1.4.0

In [ ]:
# Mount Drive + set PROJECT_ROOT
import os, sys
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/text-difficulty-classification'
except Exception:
    PROJECT_ROOT = os.path.abspath('.')
os.environ['PROJECT_ROOT'] = PROJECT_ROOT
print('PROJECT_ROOT =', PROJECT_ROOT)


Mounted at /content/drive
PROJECT_ROOT = /content/drive/MyDrive/text-difficulty-classification


In [ ]:
import glob
import os
import pickle
import re
import sys

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = os.environ['PROJECT_ROOT']
FEATURES_DIR = os.path.join(PROJECT_ROOT, 'outputs', 'features_multi')
OOD_DIR      = os.path.join(FEATURES_DIR, 'ood')
SCALERS_DIR  = os.path.join(FEATURES_DIR, 'scalers')
os.makedirs(OOD_DIR, exist_ok=True)
os.makedirs(SCALERS_DIR, exist_ok=True)

LABEL_MAP = {'elementary': 0, 'middle': 1, 'high': 2}
LABEL_COL = 'education_level'
# Must match step6b. Any non-feature column that might be in the input CSVs.
META_COLS = ['education_level', 'source_dataset', 'domain',
             'label_source', 'subject', 'split', 'raw_label', 'text_idx']


In [ ]:
def _feature_sets():
    """Discover by looking at train_*.csv."""
    out = []
    for p in sorted(glob.glob(os.path.join(FEATURES_DIR, 'train_*.csv'))):
        name = os.path.basename(p)[len('train_'):-len('.csv')]
        out.append(name)
    return out


In [ ]:
def _load_split(name, split):
    p = os.path.join(FEATURES_DIR, f'{split}_{name}.csv')
    if not os.path.exists(p):
        return None
    return pd.read_csv(p)


In [ ]:
def _list_ood_corpora(name):
    """Find every ood_<corpus>_<name>.csv for a given feature_set name."""
    pat = os.path.join(FEATURES_DIR, f'ood_*_{name}.csv')
    out = []
    for p in glob.glob(pat):
        base = os.path.basename(p)
        # ood_<corpus>_<name>.csv  →  corpus
        m = re.match(r'^ood_(.+?)_' + re.escape(name) + r'\.csv$', base)
        if m:
            out.append(m.group(1))
    return sorted(out)


In [ ]:
def _xy_from_df(df, feature_names):
    feats = df[[c for c in df.columns if c not in META_COLS]]
    feats = feats.reindex(columns=feature_names, fill_value=0.0)
    X = feats.to_numpy(dtype=np.float64)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    y_raw = df[LABEL_COL]
    if not y_raw.isin(LABEL_MAP).all():
        bad = y_raw[~y_raw.isin(LABEL_MAP)].unique()
        raise ValueError(f"unmapped labels {bad}")
    y = y_raw.map(LABEL_MAP).to_numpy()
    return X, y


In [ ]:
def _prepare(name):
    train = _load_split(name, 'train')
    val   = _load_split(name, 'val')
    test  = _load_split(name, 'test')
    if train is None:
        return None

    feature_names = [c for c in train.columns if c not in META_COLS]


    for split_name, df in (('val', val), ('test', test)):
        if df is None:
            continue
        df_feats = [c for c in df.columns if c not in META_COLS]
        missing  = sorted(set(feature_names) - set(df_feats))
        extra    = sorted(set(df_feats) - set(feature_names))
        if missing or extra:
            raise ValueError(
                f"[step7b] {name}: {split_name} columns differ from train. "
                f"Missing: {missing[:3]}... Extra: {extra[:3]}...")

    X_train, y_train = _xy_from_df(train, feature_names)

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)

    arrays = {'X_train': X_train_s, 'y_train': y_train,
              'feature_names': np.array(feature_names)}

    if val is not None:
        X_val, y_val = _xy_from_df(val, feature_names)
        arrays['X_val'], arrays['y_val'] = scaler.transform(X_val), y_val
    if test is not None:
        X_test, y_test = _xy_from_df(test, feature_names)
        arrays['X_test'], arrays['y_test'] = scaler.transform(X_test), y_test

    np.savez_compressed(os.path.join(FEATURES_DIR, f'{name}.npz'), **arrays)
    with open(os.path.join(SCALERS_DIR, f'{name}.pkl'), 'wb') as fh:
        pickle.dump({'scaler': scaler, 'feature_names': feature_names}, fh)

    # OOD splits — scaled with the same train-fit scaler.
    for corpus in _list_ood_corpora(name):
        ood = pd.read_csv(os.path.join(FEATURES_DIR, f'ood_{corpus}_{name}.csv'))
        X_ood, y_ood = _xy_from_df(ood, feature_names)
        X_ood_s = scaler.transform(X_ood)
        np.savez_compressed(
            os.path.join(OOD_DIR, f'{name}__{corpus}.npz'),
            X=X_ood_s, y=y_ood,
            source_dataset=ood.get('source_dataset', pd.Series([corpus]*len(ood))).to_numpy(),
            domain=ood.get('domain', pd.Series([''] * len(ood))).to_numpy(),
        )

    return name, arrays['X_train'].shape


In [ ]:
def main():
    sets = _feature_sets()
    if not sets:
        sys.exit(f"[step7b] no feature CSVs in {FEATURES_DIR}.")
    for s in sets:
        try:
            res = _prepare(s)
            if res:
                print(f"  {res[0]:<30} train={res[1]}")
        except Exception as e:
            print(f"  [step7b] {s} failed: {type(e).__name__}: {e}")
    print(f"[step7b] done — npz in {FEATURES_DIR}, OOD in {OOD_DIR}")


In [ ]:
main()


  all_prompts                    train=(6371, 63)
  full                           train=(6371, 89)
  fused_qwen2.5-7b               train=(6371, 89)
  prompts_qwen2.5-7b             train=(6371, 63)
  static                         train=(6371, 26)
[step7b] done — npz in /content/drive/MyDrive/text-difficulty-classification/outputs/features_multi, OOD in /content/drive/MyDrive/text-difficulty-classification/outputs/features_multi/ood
